# ROOTS — Step 3: Compound Library Curation (Python)

**Pipeline:** ROOTS — Receptor-oriented Oncotarget Screening of Natural Compounds Under Thermodynamic Simulation in Ovarian Cancer

**Input:** independent of Step 2; this step curates the natural-product screening library on its own track.

**What this notebook does:**
1. Downloads and archives the COCONUT natural-product database at retrieval time (COCONUT updates without versioning, so an unarchived pull is not reproducible)
2. Parses SMILES and applies Lipinski Ro5, Veber, and PAINS filtering via RDKit
3. Runs a stereochemistry-integrity check: defined stereocentres must survive 3D embedding
4. Deduplicates by InChIKey
5. Saves the final library to Drive for Step 4 (receptor/ligand preparation)

**Compute note:** this notebook is written to checkpoint every stage to Drive, since the stereochemistry-integrity check in particular is CPU-heavy at full library scale and has previously needed to run outside Colab's free-tier RAM ceiling (see Section 4's note). Each cell is safe to re-run from its own checkpoint without repeating earlier stages.

## 0. Environment setup and Drive mount

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd
import numpy as np
import json
from datetime import datetime

DRIVE_BASE = Path("/content/drive/MyDrive/ROOTS_Pipeline")
STEP3_DIR = DRIVE_BASE / "Step3"
STEP3_DIR.mkdir(parents=True, exist_ok=True)
(STEP3_DIR / "logs").mkdir(exist_ok=True)

print(f"Timestamp: {datetime.now().isoformat()}")
print(f"Step 3 output directory: {STEP3_DIR}")

In [ ]:
%%capture
!pip install -q rdkit requests

## 1. COCONUT database retrieval

The raw download is archived at retrieval time, with its source URL, timestamp, and a note that COCONUT updates without versioning — an unarchived pull would not be reproducible. Retrieved as `coconut_csv_lite`, the lightweight CSV export (SMILES plus core descriptors, not the full structure-file bundle), which is what the downstream RDKit filtering actually needs.

In [ ]:
import requests
import zipfile

# COCONUT's bulk CSV-lite export. The exact dated filename changes with each
# COCONUT release; confirm the current path at https://coconut.naturalproducts.net/download
# before running if this URL 404s, then update COCONUT_DOWNLOAD_URL accordingly.
COCONUT_DOWNLOAD_URL = "https://coconut.s3.uni-jena.de/prod/downloads/2026-09/coconut_csv_lite-09-2026.zip"

raw_zip_path = STEP3_DIR / "coconut_csv_lite_raw.zip"
retrieval_timestamp = datetime.now().isoformat()

if raw_zip_path.exists() and raw_zip_path.stat().st_size > 0:
    print(f"Using cached archive: {raw_zip_path} ({raw_zip_path.stat().st_size / 1e6:.1f} MB)")
else:
    print(f"Downloading COCONUT from: {COCONUT_DOWNLOAD_URL}")
    print("(This may take several minutes.)")
    resp = requests.get(COCONUT_DOWNLOAD_URL, timeout=600, stream=True)
    resp.raise_for_status()
    with open(raw_zip_path, 'wb') as f:
        for chunk in resp.iter_content(chunk_size=8192):
            f.write(chunk)
    print(f"Saved: {raw_zip_path} ({raw_zip_path.stat().st_size / 1e6:.1f} MB)")

with zipfile.ZipFile(raw_zip_path) as z:
    csv_name = [n for n in z.namelist() if n.endswith('.csv')][0]
    z.extract(csv_name, STEP3_DIR)
    raw_csv_path = STEP3_DIR / csv_name

coconut_df = pd.read_csv(raw_csv_path, low_memory=False)
print(f"\nCOCONUT loaded: {len(coconut_df):,} compounds, {len(coconut_df.columns)} columns")
print(f"Columns: {list(coconut_df.columns)[:15]}{'...' if len(coconut_df.columns) > 15 else ''}")

assert 'canonical_smiles' in coconut_df.columns, (
    f"Expected a 'canonical_smiles' column; got {list(coconut_df.columns)}. "
    "COCONUT's export schema may have changed -- inspect columns and adjust smiles_col below."
)
smiles_col = 'canonical_smiles'

retrieval_log = {
    'retrieval_timestamp': retrieval_timestamp,
    'source_url': COCONUT_DOWNLOAD_URL,
    'raw_compound_count': int(len(coconut_df)),
    'smiles_column': smiles_col,
    'note': 'COCONUT updates without versioning; this archived file is the frozen pull for this run.',
}
with open(STEP3_DIR / "logs" / "coconut_retrieval.json", 'w') as f:
    json.dump(retrieval_log, f, indent=2)
print("\nRetrieval log saved.")

## 2. SMILES parsing (chunked, checkpointed)

Parsed in chunks with a running checkpoint, so a Colab disconnect partway through loses at most one chunk's worth of work rather than the full pass.

In [ ]:
from rdkit import Chem
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

valid_checkpoint = STEP3_DIR / "coconut_valid_parsed.csv"

if valid_checkpoint.exists():
    print(f"Resuming from checkpoint: {valid_checkpoint}")
    coconut_df = pd.read_csv(valid_checkpoint, low_memory=False)
    print(f"Valid compounds loaded from checkpoint: {len(coconut_df):,}")
else:
    CHUNK_SIZE = 50000
    total = len(coconut_df)
    valid_mask = pd.Series(False, index=coconut_df.index)

    print(f"Parsing {total:,} SMILES in chunks of {CHUNK_SIZE:,}...")
    for start in range(0, total, CHUNK_SIZE):
        end = min(start + CHUNK_SIZE, total)
        chunk = coconut_df.iloc[start:end]
        parsed = chunk[smiles_col].apply(lambda s: Chem.MolFromSmiles(str(s)) is not None if pd.notna(s) else False)
        valid_mask.iloc[start:end] = parsed
        print(f"  {end:,}/{total:,}: {parsed.sum():,} valid in this chunk")

    print(f"\nValid SMILES parsed: {valid_mask.sum():,} / {total:,}")
    coconut_df = coconut_df[valid_mask].copy()
    coconut_df.to_csv(valid_checkpoint, index=False)
    print(f"Checkpoint saved: {valid_checkpoint}")

n_after_smiles_parse = len(coconut_df)

## 3. Drug-likeness filtering: Lipinski Ro5, Veber, PAINS

Applied together in one consolidated pass rather than three separate ones, to avoid recomputing RDKit mols three times over a 700K+-compound set. **Scope limitation, disclosed:** Ro5/Veber filtering applied to a natural-product library is a stated limitation, not an oversight — legitimate natural products routinely and validly violate these rules (macrocycles, glycosides, high heteroatom counts), so this step trades some true-positive natural products for a tractable, drug-likeness-biased screening set.

In [ ]:
from rdkit.Chem import Descriptors, FilterCatalog, AllChem
from rdkit.Chem.FilterCatalog import FilterCatalogParams

filtered_checkpoint = STEP3_DIR / "coconut_ro5_veber_pains_passed.csv"

if filtered_checkpoint.exists():
    print(f"Resuming from checkpoint: {filtered_checkpoint}")
    filtered_df = pd.read_csv(filtered_checkpoint, low_memory=False)
    print(f"Filtered compounds loaded from checkpoint: {len(filtered_df):,}")
else:
    params = FilterCatalogParams()
    params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS)
    pains_catalog = FilterCatalog.FilterCatalog(params)

    CHUNK_SIZE = 50000
    total = len(coconut_df)
    keep_mask = pd.Series(False, index=coconut_df.index)
    descriptor_rows = []

    print(f"Computing descriptors and applying Ro5+Veber+PAINS to {total:,} compounds...")
    for start in range(0, total, CHUNK_SIZE):
        end = min(start + CHUNK_SIZE, total)
        chunk = coconut_df.iloc[start:end]
        for idx, row in chunk.iterrows():
            mol = Chem.MolFromSmiles(str(row[smiles_col]))
            if mol is None:
                continue
            mw = Descriptors.MolWt(mol)
            logp = Descriptors.MolLogP(mol)
            hbd = Descriptors.NumHDonors(mol)
            hba = Descriptors.NumHAcceptors(mol)
            tpsa = Descriptors.TPSA(mol)
            rotb = Descriptors.NumRotatableBonds(mol)

            passes_ro5 = (mw <= 500) and (logp <= 5) and (hbd <= 5) and (hba <= 10)
            passes_veber = (tpsa <= 140) and (rotb <= 10)
            passes_pains = not pains_catalog.HasMatch(mol)

            if passes_ro5 and passes_veber and passes_pains:
                keep_mask.loc[idx] = True
                descriptor_rows.append({
                    'idx': idx, 'MW': mw, 'LogP': logp, 'HBD': hbd,
                    'HBA': hba, 'TPSA': tpsa, 'RotBonds': rotb,
                })
        print(f"  {end:,}/{total:,}: {keep_mask.iloc[start:end].sum():,} passed in this chunk")

    print(f"\nPass Ro5 + Veber + PAINS-free: {keep_mask.sum():,} / {total:,}")

    descriptors_df = pd.DataFrame(descriptor_rows).set_index('idx')
    filtered_df = coconut_df[keep_mask].copy()
    filtered_df = filtered_df.join(descriptors_df)
    filtered_df.to_csv(filtered_checkpoint, index=False)
    print(f"Checkpoint saved: {filtered_checkpoint}")

n_after_ro5_veber_pains = len(filtered_df)

## 4. Stereochemistry-integrity check on 3D embedding

Defined stereocentres in the SMILES must survive 3D embedding and MMFF optimisation. Compounds where a defined stereocentre is lost during embedding are excluded and logged, since a docking pose built on the wrong stereoisomer is worse than no pose at all.

**Compute note (disclosed):** at full library scale (300K+ compounds after the Ro5/Veber/PAINS filter), this check is CPU-heavy enough that Colab's free-tier RAM has previously been exceeded partway through. If that happens here, the same chunked, checkpointed CSV-writing pattern below is designed so the run can be moved to a CPU-only Kaggle script kernel (or any other machine) and resumed from `stereo_checkpoint.csv` without recomputation of completed chunks — just point `STEP3_DIR` at the same Drive-synced location, or copy the checkpoint file across.

In [ ]:
from rdkit.Chem import AllChem, FindMolChiralCenters

EMBED_SEED = 42
stereo_checkpoint = STEP3_DIR / "coconut_stereochem_passed.csv"
stereo_progress_path = STEP3_DIR / "logs" / "stereo_check_progress.txt"

def check_stereochemistry(mol, seed=EMBED_SEED):
    """Embed molecule in 3D and verify stereocentres survive.
    Returns (success, original_centers, embedded_centers, match)."""
    if mol is None:
        return False, [], [], False
    original_centers = FindMolChiralCenters(mol, includeUnassigned=True)
    mol_3d = Chem.AddHs(mol)
    result = AllChem.EmbedMolecule(mol_3d, AllChem.ETKDGv3())
    if result == -1:
        return False, original_centers, [], False
    try:
        AllChem.MMFFOptimizeMolecule(mol_3d, maxIters=500)
    except Exception:
        pass
    Chem.AssignStereochemistryFrom3D(mol_3d)
    embedded_centers = FindMolChiralCenters(mol_3d, includeUnassigned=True)
    if len(original_centers) == 0:
        return True, original_centers, embedded_centers, True
    orig_dict = {idx: chirality for idx, chirality in original_centers if chirality != '?'}
    embed_dict = {idx: chirality for idx, chirality in embedded_centers if chirality != '?'}
    match = all(embed_dict.get(idx) == chirality for idx, chirality in orig_dict.items())
    return True, original_centers, embedded_centers, match

if stereo_checkpoint.exists():
    print(f"Resuming from checkpoint: {stereo_checkpoint}")
    library_df = pd.read_csv(stereo_checkpoint, low_memory=False)
    print(f"Stereo-passed compounds loaded from checkpoint: {len(library_df):,}")
else:
    start_pos = int(stereo_progress_path.read_text()) if stereo_progress_path.exists() else 0
    CHUNK_SIZE = 2000
    total = len(filtered_df)
    results_so_far = []
    partial_path = STEP3_DIR / "logs" / "stereo_check_partial.csv"
    if partial_path.exists() and start_pos > 0:
        results_so_far = pd.read_csv(partial_path).to_dict('records')

    print(f"Running stereochemistry check from position {start_pos:,} / {total:,}...")
    for start in range(start_pos, total, CHUNK_SIZE):
        end = min(start + CHUNK_SIZE, total)
        chunk = filtered_df.iloc[start:end]
        for idx, row in chunk.iterrows():
            mol = Chem.MolFromSmiles(str(row[smiles_col]))
            success, orig, embed, match = check_stereochemistry(mol)
            results_so_far.append({
                'idx': idx, 'embed_success': success,
                'original_stereocenters': len(orig), 'stereo_preserved': match,
            })
        pd.DataFrame(results_so_far).to_csv(partial_path, index=False)
        stereo_progress_path.write_text(str(end))
        n_ok_so_far = sum(1 for r in results_so_far if r['embed_success'] and r['stereo_preserved'])
        print(f"  {end:,}/{total:,}: {n_ok_so_far:,} passing so far")

    stereo_df = pd.DataFrame(results_so_far).set_index('idx')
    embed_ok = stereo_df['embed_success'] & stereo_df['stereo_preserved']

    print(f"\nStereochemistry check results:")
    print(f"  Embedding successful: {stereo_df['embed_success'].sum():,} / {len(stereo_df):,}")
    print(f"  Stereocentres preserved: {stereo_df['stereo_preserved'].sum():,} / {len(stereo_df):,}")
    print(f"  Compounds passing both: {embed_ok.sum():,} / {len(stereo_df):,}")

    library_df = filtered_df.loc[stereo_df[embed_ok].index].copy()
    library_df.to_csv(stereo_checkpoint, index=False)
    print(f"\nCheckpoint saved: {stereo_checkpoint}")

n_after_stereochem = len(library_df)
print(f"\nCompounds after stereochemistry check: {n_after_stereochem:,}")

## 5. Deduplication (InChIKey)

Two COCONUT entries with different IDs can represent the same molecule (different tautomer records, database merges, etc.). Deduplication uses the full InChIKey (not just the connectivity block used for the stereoisomer-grouping analysis elsewhere in this project) to remove exact duplicates while still treating genuine stereoisomers as distinct library members.

In [ ]:
from rdkit.Chem import inchi

def compute_inchikey(smiles):
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return None
    try:
        return inchi.MolToInchiKey(mol)
    except Exception:
        return None

print(f"Computing InChIKeys for {len(library_df):,} compounds...")
library_df['inchikey'] = library_df[smiles_col].apply(compute_inchikey)

n_before_dedup = len(library_df)
duplicate_mask = library_df.duplicated(subset='inchikey', keep='first')
n_duplicates = int(duplicate_mask.sum())

duplicate_sample = library_df[duplicate_mask].head(50)
duplicate_sample.to_csv(STEP3_DIR / "logs" / "duplicate_compounds_sample.csv", index=False)

library_df = library_df[~duplicate_mask].copy()
n_final = len(library_df)

print(f"Duplicates removed (InChIKey-based): {n_duplicates:,}")
print(f"Final library size: {n_final:,}")

## 6. Save final library

In [ ]:
id_cols = [c for c in library_df.columns if 'id' in c.lower() or 'name' in c.lower() or 'coconut' in c.lower()]
output_cols = id_cols + [smiles_col, 'inchikey', 'MW', 'LogP', 'HBD', 'HBA', 'TPSA', 'RotBonds']
output_cols = [c for c in dict.fromkeys(output_cols) if c in library_df.columns]  # dedupe, preserve order

library_out = library_df[output_cols].copy()

output_path = STEP3_DIR / "library_final.csv"
library_out.to_csv(output_path, index=False)
print(f"Final library saved to: {output_path}")
print(f"  Compounds: {len(library_out):,}")
print(f"  Columns: {list(library_out.columns)}")

qc_summary = {
    'timestamp': datetime.now().isoformat(),
    'MW_mean': float(library_out['MW'].mean()) if 'MW' in library_out.columns else None,
    'MW_median': float(library_out['MW'].median()) if 'MW' in library_out.columns else None,
    'LogP_mean': float(library_out['LogP'].mean()) if 'LogP' in library_out.columns else None,
    'LogP_median': float(library_out['LogP'].median()) if 'LogP' in library_out.columns else None,
    'final_compound_count': int(len(library_out)),
}
with open(STEP3_DIR / "logs" / "step3_final_library_qc.json", 'w') as f:
    json.dump(qc_summary, f, indent=2)

attrition = {
    'timestamp': datetime.now().isoformat(),
    'coconut_raw_count': int(retrieval_log['raw_compound_count']),
    'after_smiles_parse': int(n_after_smiles_parse),
    'after_ro5_veber_pains': int(n_after_ro5_veber_pains),
    'after_stereochem': int(n_after_stereochem),
    'duplicates_removed': int(n_duplicates),
    'final_count': int(n_final),
    'seed': EMBED_SEED,
    'filters_applied': ['Lipinski Ro5', 'Veber', 'PAINS', 'Stereochemistry integrity (3D embed + MMFF)', 'InChIKey deduplication'],
    'scope_note': 'Ro5/Veber filtering applied to a natural-product library is a stated scope limitation; natural products legitimately violate these rules. No library-size cap was applied -- the full Ro5/Veber/PAINS/stereochemistry-passing, deduplicated set is retained.',
}
with open(STEP3_DIR / "logs" / "step3_attrition.json", 'w') as f:
    json.dump(attrition, f, indent=2)

print(f"\nAttrition summary:")
for k, v in attrition.items():
    print(f"  {k}: {v}")

print(f"\n=== Step 3 complete ===")
print(f"Step 4 (receptor/ligand prep) will load: {output_path}")
print(f"and structures from: {DRIVE_BASE / 'Step2'}")